# Trip Planner experiments

Offline notebook for exploring the deterministic planner and exercising the agent loop without hitting Anthropic. Provider is forced to `mock` for reproducibility.

In [1]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'src').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

os.environ.setdefault('LLM_PROVIDER', 'mock')
os.environ.setdefault('LLM_MODEL', 'mock-model')

DATA_DIR = REPO_ROOT / 'src' / 'notebook' / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT

PosixPath('/Users/omaralhasan/Desktop/Projects/case/Trip_Planner_Agent')

## 1. Deterministic plan (v0)

Calls the same tool registry the LLM uses, but assembles the day-by-day plan directly. No model calls.

In [2]:
from src.agent.planning import build_day_by_day_plan, format_plan_markdown
from src.api.models import TripPreferences

prefs = TripPreferences(
    origin='Amsterdam',
    destination='Copenhagen',
    daily_km=100,
    month='June',
    lodging_preference='camping',
    hostel_every_n_nights=4,
    nationality='Canadian',
    stay_days=10,
)

plan = build_day_by_day_plan(prefs)
print(format_plan_markdown(plan, preferences=prefs))

## Trip summary: Amsterdam → Copenhagen
- **Target pace**: ~100 km/day
- **Month**: June
- **Lodging**: camping, hostel every 4 nights
- **Budget (mock)**: ~EUR 462 total for 7 days
- **Visa note (mock)**: may_be_required — Mock result. Verify officially.

## Day-by-day plan
### Day 1: Amsterdam → Day1_Stop
- **Distance**: 102.0 km
- **Terrain**: 1166 m gain (moderate)
- **Weather**: Typical June weather: mild to warm, with a chance of rain. (avg 13–20°C)
- **Sleep**: Camping: Day1_Stop Camping 1 (~€29)
- **Highlights**: Day1_Stop Food #1, Day1_Stop Food #2, Day1_Stop Food #3, Day1_Stop Food #4

### Day 2: Day1_Stop → Day2_Stop
- **Distance**: 102.0 km
- **Terrain**: 1176 m gain (moderate)
- **Weather**: Typical June weather: mild to warm, with a chance of rain. (avg 19–27°C)
- **Sleep**: Camping: Day2_Stop Camping 1 (~€19)
- **Highlights**: Day2_Stop Food #1, Day2_Stop Food #2, Day2_Stop Food #3, Day2_Stop Food #4

### Day 3: Day2_Stop → Day3_Stop
- **Distance**: 102.0 km
- **Terrain*

## 2. Snapshot the plan

Pin a deterministic output so we can diff against future changes to the planner or mock data.

In [3]:
import json

snapshot_path = DATA_DIR / 'deterministic_example.json'
snapshot_path.write_text(
    json.dumps(
        {
            'preferences': prefs.model_dump(),
            'days': [d.model_dump() for d in plan],
        },
        indent=2,
    ),
    encoding='utf-8',
)
snapshot_path

PosixPath('/Users/omaralhasan/Desktop/Projects/case/Trip_Planner_Agent/src/notebook/data/deterministic_example.json')

## 3. Variant sweep

Tiny eval harness: change one preference at a time, check the resulting day count, total km, and that the formatter still produces a budget + visa line.

In [ ]:
import pandas as pd

variants = {
    'baseline (100 km, camping)': prefs,
    'slow (80 km/day)': prefs.model_copy(update={'daily_km': 80}),
    'mixed lodging': prefs.model_copy(update={'lodging_preference': 'mixed'}),
    'July traveller': prefs.model_copy(update={'month': 'July'}),
}

rows = []
for label, p in variants.items():
    days = build_day_by_day_plan(p)
    md = format_plan_markdown(days, preferences=p)
    rows.append({
        'variant': label,
        'days': len(days),
        'total_km': round(sum(d.distance_km for d in days), 1),
        'has_budget': '**Budget**' in md,
        'has_visa': '**Visa note' in md,
    })

pd.DataFrame(rows)

## 4. Agent loop trace (scripted mock)

Drives the real `run_agent_loop` against a scripted mock so we can see the tool-use sequence without an Anthropic key.

In [5]:
from src.agent.orchestration import run_agent_loop
from src.agent.providers.mock_provider import MockProvider, MockResponse, text_block, tool_use_block
from src.tools.builtins import build_registry

registry = build_registry()
scripted = [
    MockResponse(
        content=[tool_use_block('a', 'get_route', {'origin': 'Amsterdam', 'destination': 'Copenhagen'})],
        stop_reason='tool_use',
    ),
    MockResponse(
        content=[
            tool_use_block('b', 'get_weather', {'location': 'Copenhagen', 'month': 'June'}),
            tool_use_block('c', 'find_accommodation', {'near': 'Copenhagen', 'kind': 'camping'}),
        ],
        stop_reason='tool_use',
    ),
    MockResponse(
        content=[text_block('Final plan would go here.')],
        stop_reason='end_turn',
    ),
]
provider = MockProvider(responses=scripted)

result = run_agent_loop(
    messages=[{'role': 'user', 'content': 'Plan Amsterdam to Copenhagen'}],
    registry=registry,
    provider=provider,
    max_rounds=6,
    max_tokens=1024,
)

{
    'rounds': result.rounds,
    'truncated': result.truncated,
    'tool_calls': [t.name for t in result.tool_calls],
    'reply': result.reply,
}

{'rounds': 3,
 'truncated': False,
 'tool_calls': ['get_route', 'get_weather', 'find_accommodation'],
 'reply': 'Final plan would go here.'}

In [6]:
from src.api.app import app
from fastapi.testclient import TestClient

client = TestClient(app)
r = client.post('/api/v1/chat', json={'message': 'Plan Amsterdam to Copenhagen, 100km/day, camping, June'})
data = r.json()
{
    'status': r.status_code,
    'rounds': data['rounds'],
    'tool_calls': [t['name'] for t in data['tool_calls']],
    'reply_preview': (data.get('reply') or '')[:200],
}

{'status': 200, 'rounds': 1, 'tool_calls': [], 'reply_preview': 'OK (mock).'}